In [76]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# csv読み込み
df = pd.read_csv("stylometry_features.csv")

df = df.fillna(0)

y = df["label"]

all_features = [
    col for col in df.columns
    if col not in ["filename", "label"]
]

function_features = [
    "機能語率",
    "ひらがな率",
    "文長標準偏差",
    "平均文長",
    "最大文長",
    "句点頻度",
    "読点頻度",
    "助詞割合",
    "助動詞割合",
    "連体詞割合",
    "接続詞割合",
    "感動詞割合",
    
    "Hapax率_func",
    "Yules_K_func",

    "助詞_は",
    "助詞_に",
    "助詞_て",
    "助詞_と",
    "助詞_か",
    "助詞_が",
    "助詞_で",
    "助詞_や",
    "助詞_を",
    "助詞_の",
    "助詞_ほど",
    "助詞_まで",
    "助詞_も",
    "助詞_から",
    "助詞_たり",
    "助詞_など",
    "助詞_ながら",
    "助詞_だけ",
    "助詞_ば",
    "助詞_へ",
    "助詞_のみ",
    "助詞_しも",
    "助詞_しか",
    "助詞_すら",
    "助詞_より",
    "助詞_こそ",
    "助詞_つつ",
    "助詞_くらい",
    "助詞_かしら",
    "助詞_ずつ"
]

content_features = [
    "内容語率",
    "総単語数",
    "異なり語数",
    "TTR",
    # "カタカナ率",
    # "漢字率",
    # "英数字率",

    # "Hapax率_cont",
    "Yules_K_cont",

    # "名詞割合",
    # "動詞割合",
    "形容詞割合",
    "副詞割合",
    "形状詞割合",

    # "感嘆符数",
    # "疑問符数"
]

features_sets = {
    "All": all_features,
    "Function": function_features,
    "Content": content_features
}

coef_results = {}

for name, features in features_sets.items():
    print("\n========")
    print(name)
    print("========")
    
    X = df[features]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(max_iter=5000, class_weight="balanced")

    model.fit(X_train_scaled, y_train)
    
    coef = pd.DataFrame({
        "features": features,
        "coef": model.coef_[0]
    })
    
    coef["abs_coef"] = coef["coef"].abs()
    
    coef_results[name] = coef
    
    y_pred = model.predict(X_test_scaled)
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Accuracy：", accuracy_score(y_test, y_pred))


All
              precision    recall  f1-score   support

          AI       0.86      1.00      0.92         6
       Human       1.00      0.83      0.91         6

    accuracy                           0.92        12
   macro avg       0.93      0.92      0.92        12
weighted avg       0.93      0.92      0.92        12

Accuracy： 0.9166666666666666

Function
              precision    recall  f1-score   support

          AI       1.00      1.00      1.00         6
       Human       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12

Accuracy： 1.0

Content
              precision    recall  f1-score   support

          AI       0.71      0.83      0.77         6
       Human       0.80      0.67      0.73         6

    accuracy                           0.75        12
   macro avg       0.76      0.75      0.75        12
weighte

In [72]:
from sklearn.model_selection import cross_val_score

for name, features in features_sets.items():
    
    X = df[features]
    
    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring ="accuracy"
    )
    
    print("-------")
    print(name)
    print("--------")
    print(scores)
    print("平均：", scores.mean())

-------
All
--------
[0.91666667 0.91666667 0.75       0.75       0.91666667]
平均： 0.85
-------
Function
--------
[0.66666667 0.91666667 0.75       0.75       0.75      ]
平均： 0.7666666666666666
-------
Content
--------
[0.75       0.75       0.83333333 0.75       0.75      ]
平均： 0.7666666666666667


In [47]:
for name, coef in coef_results.items():
    print("\n--------")
    print(name)
    print("--------")
    print(
        coef.sort_values(
            "abs_coef",
            ascending=False
        )
    )


--------
All
--------
   features      coef  abs_coef
35     助詞_や -0.808914  0.808914
24     副詞割合  0.508814  0.508814
21    形状詞割合 -0.500823  0.500823
33     助詞_が  0.425147  0.425147
9    文長標準偏差  0.408914  0.408914
..      ...       ...       ...
28     助詞_は  0.014489  0.014489
29     助詞_に -0.008639  0.008639
57    助詞_こそ  0.006258  0.006258
40     感嘆符数  0.000000  0.000000
41     疑問符数  0.000000  0.000000

[62 rows x 3 columns]

--------
Function
--------
        features      coef  abs_coef
21          助詞_や -1.012849  1.012849
2         文長標準偏差  0.603460  0.603460
7           助詞割合  0.588159  0.588159
9          連体詞割合 -0.539716  0.539716
19          助詞_が  0.526230  0.526230
18          助詞_か  0.511354  0.511354
6           読点頻度 -0.453297  0.453297
22          助詞_を -0.428035  0.428035
27         助詞_から  0.404021  0.404021
12   Hapax率_func  0.374543  0.374543
31         助詞_だけ -0.333769  0.333769
0           機能語率  0.328895  0.328895
37         助詞_すら  0.318234  0.318234
23          助詞_の  0.3054

In [78]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# ===========================
# CSV読み込み
# ===========================
train_df = pd.read_csv("stylometry_features_train.csv")
test_df = pd.read_csv("stylometry_features_test.csv")

train_df = train_df.fillna(0)
test_df = test_df.fillna(0)

# 学習データにある列でテストデータにないものを追加
for col in train_df.columns:
    if col not in test_df.columns:
        test_df[col] = 0

# 列の順番を学習データと揃える
test_df = test_df[train_df.columns]

# ===========================
# 特徴量
# ===========================
all_features = [
    col for col in train_df.columns
    if col not in ["filename", "label"]
]

function_features = [
    "機能語率",
    "ひらがな率",
    "文長標準偏差",
    "平均文長",
    "最大文長",
    "句点頻度",
    "読点頻度",
    "助詞割合",
    "助動詞割合",
    "連体詞割合",
    "接続詞割合",
    "感動詞割合",

    "Hapax率_func",
    "Yules_K_func",

    "助詞_は",
    "助詞_に",
    "助詞_て",
    "助詞_と",
    "助詞_か",
    "助詞_が",
    "助詞_で",
    "助詞_や",
    "助詞_を",
    "助詞_の",
    "助詞_ほど",
    "助詞_まで",
    "助詞_も",
    "助詞_から",
    "助詞_たり",
    "助詞_など",
    "助詞_ながら",
    "助詞_だけ",
    "助詞_ば",
    "助詞_へ",
    "助詞_のみ",
    "助詞_しも",
    "助詞_しか",
    "助詞_すら",
    "助詞_より",
    "助詞_こそ",
    "助詞_つつ",
    "助詞_くらい",
    "助詞_かしら",
    "助詞_ずつ"
]

content_features = [
    "内容語率",
    "総単語数",
    "異なり語数",
    "TTR",
    "カタカナ率",
    "漢字率",
    "英数字率",

    "Hapax率_cont",
    "Yules_K_cont",

    "名詞割合",
    "動詞割合",
    "形容詞割合",
    "副詞割合",
    "形状詞割合",

    "感嘆符数",
    "疑問符数"
]

features_sets = {
    "All": all_features,
    "Function": function_features,
    "Content": content_features
}

coef_results = {}

# ===========================
# 学習・評価
# ===========================
for name, features in features_sets.items():

    print("\n========")
    print(name)
    print("========")

    X_train = train_df[features]
    y_train = train_df["label"]

    X_test = test_df[features]
    y_test = test_df["label"]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(
        max_iter=5000,
        class_weight="balanced"
    )

    model.fit(X_train_scaled, y_train)

    coef = pd.DataFrame({
        "features": features,
        "coef": model.coef_[0]
    })

    coef["abs_coef"] = coef["coef"].abs()

    coef_results[name] = coef

    y_pred = model.predict(X_test_scaled)

    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    print("Accuracy:", accuracy_score(
        y_test,
        y_pred
    ))


All
              precision    recall  f1-score   support

          AI       0.83      1.00      0.91        10
       Human       1.00      0.80      0.89        10

    accuracy                           0.90        20
   macro avg       0.92      0.90      0.90        20
weighted avg       0.92      0.90      0.90        20

Accuracy: 0.9

Function
              precision    recall  f1-score   support

          AI       0.83      1.00      0.91        10
       Human       1.00      0.80      0.89        10

    accuracy                           0.90        20
   macro avg       0.92      0.90      0.90        20
weighted avg       0.92      0.90      0.90        20

Accuracy: 0.9

Content
              precision    recall  f1-score   support

          AI       0.75      0.60      0.67        10
       Human       0.67      0.80      0.73        10

    accuracy                           0.70        20
   macro avg       0.71      0.70      0.70        20
weighted avg       0.7

In [82]:
# ===========================
# 係数表示・CSV保存
# ===========================
for name, coef in coef_results.items():

    # 絶対値順
    coef_abs = coef.sort_values(
        "abs_coef",
        ascending=False
    )

    # AI側（正の係数）
    coef_positive = coef.sort_values(
        "coef",
        ascending=False
    )

    # Human側（負の係数）
    coef_negative = coef.sort_values(
        "coef",
        ascending=True
    )

    print("\n===================")
    print(name)
    print("===================")
    print("【絶対値 上位10】")
    print(coef_abs.head(10))

    print("\n【AI側に寄与 上位10】")
    print(coef_positive.head(10))

    print("\n【Human側に寄与 上位10】")
    print(coef_negative.head(10))

    # CSV保存
    coef_abs.to_csv(
        f"coef_abs_{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    coef_positive.to_csv(
        f"coef_positive_{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

    coef_negative.to_csv(
        f"coef_negative_{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )


All
【絶対値 上位10】
   features      coef  abs_coef
35     助詞_や -0.892462  0.892462
9    文長標準偏差  0.480868  0.480868
38     読点頻度 -0.479527  0.479527
21    形状詞割合 -0.466605  0.466605
25    連体詞割合 -0.462430  0.462430
33     助詞_が  0.456578  0.456578
24     副詞割合  0.439693  0.439693
43    助詞_まで  0.413978  0.413978
45    助詞_から  0.409880  0.409880
32     助詞_か  0.377341  0.377341

【AI側に寄与 上位10】
       features      coef  abs_coef
9        文長標準偏差  0.480868  0.480868
33         助詞_が  0.456578  0.456578
24         副詞割合  0.439693  0.439693
43        助詞_まで  0.413978  0.413978
45        助詞_から  0.409880  0.409880
32         助詞_か  0.377341  0.377341
22         助詞割合  0.354046  0.354046
13  Hapax率_func  0.331801  0.331801
10         最大文長  0.246406  0.246406
12  Yules_K_all  0.239995  0.239995

【Human側に寄与 上位10】
   features      coef  abs_coef
35     助詞_や -0.892462  0.892462
38     読点頻度 -0.479527  0.479527
21    形状詞割合 -0.466605  0.466605
25    連体詞割合 -0.462430  0.462430
49    助詞_だけ -0.373823  0.373823
36     助詞_を